In [ ]:
# Install all Python dependencies required by this notebook and the RAG ingestion pipeline.
# The "-q" flag keeps pip output concise.
# The requirements file is expected to be available in the current working directory.
%pip install -q -r requirements.txt


In [ ]:
# Import the helper used to load values from a local .env file into the process environment.
from dotenv import load_dotenv

# Read environment variables from the .env file, if one exists.
load_dotenv()


In [ ]:
# Import os so we can read and define environment variables used by the ingestion pipeline.
import os

# Define the default target size for chunks when the caller has not configured it elsewhere.
# This value is measured in tokens, not characters.
os.environ.setdefault("CHUNK_TOKEN_TARGET", "1024")

# Define the minimum fraction of a page that an extracted figure should occupy to be retained.
os.environ.setdefault("FIGURE_AREA_THRESHOLD", "0.01")

# Define the directory where extraction and ingestion reports should be written.
os.environ.setdefault("REPORT_DIR", "reports")


In [ ]:
# Import Path so PDF paths can be handled consistently across operating systems.
from pathlib import Path

# Import the ingestion modules that contain configuration, parsing, chunking, and table helpers.
from rag import chunking, clients, config, docling_io, tables

# Point the notebook at the PDF that will be parsed and ingested.
SOURCE_PDF = Path("pdfs/AI-Enablers-Adopters-research-report.pdf")

# Set this to a cloud bucket name when figure images and parse artifacts should be shared remotely.
# None means that figure images and cache artifacts remain local.
BUCKET = None

# Create a stable document identifier from the PDF filename.
# This identifier is later stored in record metadata and used to scope record IDs.
DOC_ID = config.slugify(SOURCE_PDF.stem)

# Print the key configuration values so the run can be verified before processing starts.
print(f"document   : {DOC_ID}")
print(f"embedding  : {config.EMBED_MODEL} ({clients.EMBED_DIMS}d)")
print(f"chunk size : {config.CHUNK_TOKENS} tokens")
print(f"vision     : {config.VISION_MODEL}")
print(f"reports    : {config.REPORT_DIR.resolve()}")


In [ ]:
# Parse the source PDF with the project's Docling-based parser.
# The result is the structured document object used by all later ingestion stages.
doc = docling_io.parse_pdf(SOURCE_PDF)


In [ ]:
# Export the parsed document to markdown once.
# Rendering the entire document can be expensive, especially for long PDFs,
# so the markdown is reused for both inspection and the document-date heuristic.
markdown = doc.export_to_markdown()

# Infer the document's publication/report date from the first 4,000 characters of the document.
# This is intentionally separate from the ingestion timestamp.
doc_date = chunking.document_date(SOURCE_PDF, markdown[:4000])

# Print basic parsing/date information so the run can be sanity-checked.
print(f"pages {len(doc.pages)}   document date {doc_date}")


In [ ]:
# Import the heading-cleaning pass that runs before HybridChunker.
# It corrects SectionHeaderItems that are actually captions, sentence fragments,
# overly long sentence-like text, or labels inside exhibits.
from rag.headings import clean_headings

# Mutate the parsed document so false headings become ordinary TextItems.
# This must happen before HybridChunker because headings determine chunk boundaries.
clean_headings(doc)


In [ ]:
# Import the configured chunk-token target.
from backup.RAG.aws.rag.config import CHUNK_TOKENS

# Import the tokenizer encoding used by the embedding model.
from rag.config import ENCODING

# Import Docling's HybridChunker, which performs structure-aware and token-budget chunking.
from docling.chunking import HybridChunker

# Import the serializer classes needed to customize how document elements are serialized.
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer, ChunkingSerializerProvider,
)

# Import the tokenizer adapter used by HybridChunker.
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer

# Import a serializer that preserves table structure as markdown.
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer


# Define a custom serializer provider so tables are serialized as markdown during chunking.
class MarkdownTableProvider(ChunkingSerializerProvider):
    # HybridChunker calls get_serializer with "doc" as a keyword argument,
    # so the parameter name must remain exactly "doc".
    # **kwargs makes the provider tolerant of additional arguments in future Docling versions.
    def get_serializer(self, doc, **kwargs):
        # Build the serializer for this document and explicitly use markdown for tables.
        return ChunkingDocSerializer(
            doc=doc, table_serializer=MarkdownTableSerializer())


In [ ]:
# Create the HybridChunker using the embedding model's tokenizer and the configured token budget.
chunker = HybridChunker(
    # Use the same tokenizer encoding as the embedding model so token counts match embedding limits.
    tokenizer=OpenAITokenizer(tokenizer=ENCODING, max_tokens=CHUNK_TOKENS),

    # Use the custom provider so table content is represented as markdown rather than flattened text.
    serializer_provider=MarkdownTableProvider(),

    # Disable Docling's peer merging because this project performs its own type-aware prose merging later.
    merge_peers=False,
)


In [ ]:
# Run HybridChunker over the cleaned document.
# The result is converted to a list because the later passes iterate over the chunks multiple times.
chunks = list(chunker.chunk(dl_doc=doc))


In [ ]:
# Extract and persist figure images associated with the parsed document.
# The returned dictionary maps Docling picture references to stored image URIs.
uris = docling_io.save_figures(doc, DOC_ID, BUCKET)

# Report how many figure images were successfully stored.
print(f"{len(uris)} figure images stored")


In [ ]:
# Import the first project-specific chunking pass.
from rag.chunking import _to_entries

# Convert Docling chunks into lightweight entries.
# This pass classifies content, removes small page-furniture text, and marks figure-only chunks as figure slots.
entries, dropped = _to_entries(chunks, chunker, uris)


In [ ]:
# Import the prose-merging pass.
from rag.chunking import _merge_prose

# Merge compatible adjacent prose entries under the same heading.
# The merge is type-aware and avoids welding tables/figures into prose records.
entries, merges = _merge_prose(entries)


In [ ]:
# Import the optional minimum-size floor pass.
from rag.chunking import _apply_floor

# If MIN_CHUNK_TOKENS is configured above zero, small text records can absorb the next text record.
# This is the only chunk-shaping pass that is allowed to cross a heading boundary.
entries, floor_merges = _apply_floor(entries)


In [ ]:
# Import the conversion function and record-ID factory.
from rag.chunking import _to_records, _record_factory

# Build a lookup from every Docling element reference to its original document item.
# This is needed later to resolve figure and table references back to their source elements.
items_by_ref = {}

# Walk through all document elements in document order.
for item, _ in doc.iterate_items():
    # Read the stable Docling self-reference for this element.
    ref = getattr(item, "self_ref", None)

    # Only store elements that actually have a reference.
    if ref:
        items_by_ref[ref] = item

# Create the record factory for this document.
# The factory owns the occurrence counter used to keep identical repeated text as separate records.
make = _record_factory(SOURCE_PDF, DOC_ID, doc_date)

# Convert entries into embedding-ready records.
# Figure slots expand into one record per figure, while table fragments are grouped by logical table.
records, table_groups, fig_stats = _to_records(
    entries, doc, items_by_ref, uris, make)


In [ ]:
# Import the table serializer used to reconstruct complete logical tables from the document.
from rag.tables import table_markdown

# Import the pass that creates searchable table-summary records.
from rag.chunking import _table_summaries

# Serialize each logical table from the complete document structure.
tables = table_markdown(doc)

# Create one additional summary record for each table that needs a summary.
# The original table fragments remain in records because they contain exact values.
summaries, table_stats = _table_summaries(
    table_groups, tables, doc, items_by_ref, make)

# Append the table summaries to the existing records.
records += summaries


In [ ]:
# Import the finalization pass.
from rag.chunking import _finalise

# Put all records into final reading order, add prev/next links,
# bound metadata size, and truncate only unsplittable oversized records.
records, truncated = _finalise(records)


In [ ]:
# Import the diagnostic reporting function.
from rag.chunking import _report

# Print a complete ingestion/chunking report.
# Combine statistics collected from the figure, table, furniture, prose-merge,
# minimum-floor, and truncation passes into one dictionary for reporting.
_report(
    records,
    chunks,
    tables,
    table_groups,
    {
        **fig_stats,
        **table_stats,
        "dropped": dropped,
        "merges": merges,
        "floor_merges": floor_merges,
        "truncated": truncated,
    },
)
